# Plant Recognition V4 — Auto End-to-End Colab

**Mục tiêu:** chạy một notebook duy nhất để:
1. audit Botanical Vision;
2. tự động chọn 15–20 species có dữ liệu mạnh;
3. tạo dataset cân bằng;
4. train nhiều backbone phù hợp T4;
5. đánh giá internal test + robustness;
6. chọn model tốt nhất theo validation/test;
7. export ONNX + metadata;
8. chạy thử inference trên video với temporal smoothing.

**Không ép accuracy.** Notebook báo kết quả thật. Mục tiêu 97–99% chỉ được coi là đạt nếu đo được trên tập test độc lập.

Nguồn dataset: `dbabnigg/botanical-vision-256` — 407,759 ảnh, 4,094 species; train/val/test 70/15/15 theo species; duplicate được loại trước khi split; ảnh nguồn từ iNaturalist Research-grade qua GBIF.

Dataset card: https://huggingface.co/datasets/dbabnigg/botanical-vision-256


In [ ]:
!pip -q install -U datasets huggingface_hub timm onnx onnxruntime-gpu albumentations scikit-learn seaborn pyarrow fastparquet fsspec aiohttp


In [ ]:
import os, gc, json, math, time, random, shutil, warnings, re, io, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

ROOT = Path("/content/plant_recognition_v4")
for d in ["metadata","data","models","reports","exports","camera_test"]:
    (ROOT/d).mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else 0
print("Device:", DEVICE)
print("GPU:", GPU_NAME)
print("VRAM:", round(VRAM_GB,2), "GB")

# Auto configuration for a T4-class Colab.
if VRAM_GB >= 14:
    BATCH_B0, BATCH_B2, BATCH_V2S = 64, 32, 16
elif VRAM_GB >= 10:
    BATCH_B0, BATCH_B2, BATCH_V2S = 48, 24, 12
else:
    BATCH_B0, BATCH_B2, BATCH_V2S = 32, 16, 8

NUM_WORKERS = min(4, os.cpu_count() or 2)
print("Batch sizes:", BATCH_B0, BATCH_B2, BATCH_V2S)


## 1. Audit Botanical Vision metadata

Bước này **không tải toàn bộ ảnh vào RAM**. Notebook lấy metadata từ Parquet để biết species nào có đủ dữ liệu trước khi tải ảnh cần thiết.


In [ ]:
import requests, pyarrow.parquet as pq, fsspec

DATASET = "dbabnigg/botanical-vision-256"
API = "https://datasets-server.huggingface.co"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=90)
    r.raise_for_status()
    return r.json()

parq = get_json(f"{API}/parquet", {"dataset": DATASET})
json.dump(parq, open(ROOT/"metadata/parquet_files.json","w"), indent=2)

print("Parquet files:")
for x in parq.get("parquet_files", []):
    print(x["split"], x["filename"], round(x["size"]/1e9,2), "GB")

meta_frames = []
for pf in parq.get("parquet_files", []):
    split = pf["split"]
    url = pf["url"]
    print("Reading metadata:", split)
    fs = fsspec.filesystem("http")
    with fs.open(url, "rb") as f:
        table = pq.read_table(
            f,
            columns=["label","species","species_key","genus","family","order","class"]
        )
    df = table.to_pandas()
    df["split"] = split
    meta_frames.append(df)

meta = pd.concat(meta_frames, ignore_index=True)
meta.to_parquet(ROOT/"metadata/all_metadata.parquet", index=False)

counts = (
    meta.groupby(["species","species_key","genus","family","split"], dropna=False)
        .size().unstack(fill_value=0).reset_index()
)
for c in ["train","validation","test"]:
    if c not in counts.columns:
        counts[c] = 0
counts["total"] = counts[["train","validation","test"]].sum(axis=1)

print("Rows:", len(meta))
print("Species:", meta["species"].nunique())
print(counts["total"].describe())


## 2. Tự động chọn class

Heuristic:
- đủ ít nhất 700 ảnh train nếu có thể;
- ưu tiên nhiều dữ liệu;
- phạt lặp cùng genus/family để tăng diversity;
- chọn tối đa 20 class, nhưng nếu dataset thực tế không có đủ class mạnh thì tự giảm xuống.

Notebook lưu toàn bộ bảng candidate để có thể kiểm tra sau.


In [ ]:
TARGET_CLASSES = 20
MIN_TOTAL = 700
MIN_TRAIN = 450

cand = counts[
    (counts["total"] >= MIN_TOTAL) &
    (counts["train"] >= MIN_TRAIN)
].copy()

cand["base_score"] = np.log1p(cand["train"]) * 2.0 + np.log1p(cand["total"])

selected = []
genus_used, family_used = {}, {}

for _ in range(TARGET_CLASSES):
    if cand.empty:
        break
    def score(row):
        s = row.base_score
        s -= 2.5 * genus_used.get(row.genus, 0)
        s -= 1.2 * family_used.get(row.family, 0)
        return s
    cand["_score"] = cand.apply(score, axis=1)
    r = cand.loc[cand["_score"].idxmax()]
    selected.append(r.to_dict())
    genus_used[r.genus] = genus_used.get(r.genus, 0) + 1
    family_used[r.family] = family_used.get(r.family, 0) + 1
    cand = cand.drop(index=r.name)

selected_df = pd.DataFrame(selected).sort_values("train", ascending=False).reset_index(drop=True)
if len(selected_df) < 15:
    raise RuntimeError(
        f"Chỉ tìm được {len(selected_df)} class đủ dữ liệu. "
        "Giảm MIN_TOTAL/MIN_TRAIN nếu muốn nhiều class hơn."
    )

selected_df.to_csv(ROOT/"metadata/selected_classes.csv", index=False)
json.dump(selected_df.to_dict("records"), open(ROOT/"metadata/selected_classes.json","w"), indent=2)

print(selected_df[[
    "species","genus","family","train","validation","test","total"
]].to_string(index=False))


## 3. Tải ảnh đã chọn

Notebook ưu tiên đọc Parquet theo row-group và chỉ lấy các species đã chọn. Nếu backend Parquet không hỗ trợ lọc hiệu quả, nó vẫn hoàn tất bằng cách đọc các row-group cần thiết.

Mục tiêu mỗi class:
- train: tối đa 1,000 ảnh;
- validation: tối đa 200;
- test: tối đa 200.

Nếu class ít hơn thì lấy toàn bộ dữ liệu có sẵn.


In [ ]:
from collections import defaultdict

selected_species = set(selected_df["species"].tolist())
TARGET_PER_SPLIT = {"train":1000, "validation":200, "test":200}

for split in ["train","validation","test"]:
    for sp in selected_species:
        (ROOT/"data"/split/sp).mkdir(parents=True, exist_ok=True)

def image_to_pil(obj):
    # HF Parquet image feature is normally a struct with bytes/path.
    if isinstance(obj, dict):
        b = obj.get("bytes")
        if b is not None:
            return Image.open(io.BytesIO(b)).convert("RGB")
        p = obj.get("path")
        if p:
            return Image.open(p).convert("RGB")
    if isinstance(obj, (bytes, bytearray)):
        return Image.open(io.BytesIO(obj)).convert("RGB")
    if isinstance(obj, Image.Image):
        return obj.convert("RGB")
    raise TypeError(f"Unsupported image value: {type(obj)}")

saved = defaultdict(int)

for pf in parq.get("parquet_files", []):
    split = pf["split"]
    if split not in TARGET_PER_SPLIT:
        continue

    url = pf["url"]
    print("\nProcessing", split)

    fs = fsspec.filesystem("http")
    with fs.open(url, "rb") as f:
        pfobj = pq.ParquetFile(f)

        for rg in range(pfobj.num_row_groups):
            # Read species first for this row-group; avoid image bytes if no target species exists.
            tmeta = pfobj.read_row_group(rg, columns=["species"])
            s = np.array(tmeta["species"].to_pylist(), dtype=object)
            if not np.isin(s, list(selected_species)).any():
                continue

            # Read target rows with image + species.
            tab = pfobj.read_row_group(rg, columns=["species","image"])
            species_col = tab["species"].to_pylist()
            image_col = tab["image"].to_pylist()

            for i, sp in enumerate(species_col):
                if sp not in selected_species:
                    continue
                if saved[(split,sp)] >= TARGET_PER_SPLIT[split]:
                    continue
                try:
                    im = image_to_pil(image_col[i])
                    idx = saved[(split,sp)]
                    im.save(ROOT/"data"/split/sp/f"{idx:05d}.jpg", quality=95)
                    saved[(split,sp)] += 1
                except Exception as e:
                    print("skip", split, sp, e)

print("\nSaved:")
for sp in sorted(selected_species):
    print(
        sp,
        "train=", saved[("train",sp)],
        "val=", saved[("validation",sp)],
        "test=", saved[("test",sp)]
    )

pd.DataFrame([
    {"species":sp,"split":split,"count":saved[(split,sp)]}
    for sp in sorted(selected_species)
    for split in ["train","validation","test"]
]).to_csv(ROOT/"metadata/saved_counts.csv", index=False)


## 4. DataLoader + augmentation camera-oriented

Augmentation mô phỏng một phần điều kiện camera:
- crop;
- rotation;
- brightness/contrast;
- blur;
- JPEG-like degradation;
- random erasing.

Validation/test giữ preprocessing cố định.


In [ ]:
IMAGENET_MEAN = [0.485,0.456,0.406]
IMAGENET_STD = [0.229,0.224,0.225]

def make_transforms(size=224):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.60,1.0), ratio=(0.75,1.33)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.04),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1,2.0))], p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.20, scale=(0.02,0.15), ratio=(0.3,3.3))
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(size*256/224)),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])
    return train_tf, eval_tf

NUM_CLASSES = len(selected_species)
class_names = sorted(selected_species)
class_to_idx = {c:i for i,c in enumerate(class_names)}

json.dump(
    {"class_to_idx":class_to_idx, "idx_to_class":{str(v):k for k,v in class_to_idx.items()}},
    open(ROOT/"metadata/class_mapping_v4.json","w"), indent=2
)

print("Classes:", NUM_CLASSES)
print("\n".join(f"{i}: {c}" for i,c in enumerate(class_names)))


In [ ]:
def build_loaders(size, batch):
    train_tf, eval_tf = make_transforms(size)
    train_ds = datasets.ImageFolder(ROOT/"data"/"train", transform=train_tf)
    val_ds   = datasets.ImageFolder(ROOT/"data"/"validation", transform=eval_tf)
    test_ds  = datasets.ImageFolder(ROOT/"data"/"test", transform=eval_tf)

    # ImageFolder sorts class directories alphabetically; assert mapping.
    assert train_ds.class_to_idx == class_to_idx
    assert val_ds.class_to_idx == class_to_idx
    assert test_ds.class_to_idx == class_to_idx

    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)
    val_loader = DataLoader(val_ds, batch_size=batch, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)
    test_loader = DataLoader(test_ds, batch_size=batch, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_loaders(224, BATCH_B0)
print("train:", len(train_loader.dataset), "val:", len(val_loader.dataset), "test:", len(test_loader.dataset))


## 5. Train/evaluate nhiều backbone

Thử:
- EfficientNet-B0;
- EfficientNet-B2;
- EfficientNetV2-S nếu VRAM cho phép.

Nếu model nặng gây CUDA OOM, notebook tự giảm batch và thử lại. Mỗi model dùng early stopping và lưu best checkpoint.


In [ ]:
from copy import deepcopy

def make_model(name, nclasses):
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, nclasses)
    elif name == "efficientnet_b2":
        m = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, nclasses)
    elif name == "efficientnet_v2_s":
        m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, nclasses)
    else:
        raise ValueError(name)
    return m

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred, probs_all = [], [], []
    loss_fn = nn.CrossEntropyLoss()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE, non_blocking=True),y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE=="cuda"):
                logits=model(x)
                loss=loss_fn(logits,y)
            p=torch.softmax(logits,dim=1)
            total_loss += loss.item()*x.size(0); n += x.size(0)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(p.argmax(1).cpu().numpy().tolist())
            probs_all.append(p.cpu().numpy())
    probs=np.concatenate(probs_all,axis=0)
    return {
        "loss": total_loss/n,
        "accuracy": accuracy_score(y_true,y_pred),
        "macro_f1": f1_score(y_true,y_pred,average="macro"),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs
    }

def train_one(name, epochs=25, patience=6, base_lr=2e-4):
    if name=="efficientnet_b0":
        size,batch=224,BATCH_B0
    elif name=="efficientnet_b2":
        size,batch=260,BATCH_B2
    else:
        size,batch=224,BATCH_V2S

    try:
        tr,va,te=build_loaders(size,batch)
    except Exception as e:
        print("loader error",name,e); return None

    model=make_model(name,NUM_CLASSES).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=base_lr,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    loss_fn=nn.CrossEntropyLoss(label_smoothing=0.10)
    scaler=torch.amp.GradScaler("cuda", enabled=DEVICE=="cuda")

    best_acc=-1; best_state=None; bad=0; history=[]
    ckpt_path=ROOT/"models"/f"{name}_best.pth"

    for ep in range(1,epochs+1):
        model.train()
        run_loss=0; correct=0; total=0
        t0=time.time()
        try:
            for x,y in tr:
                x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=DEVICE=="cuda"):
                    logits=model(x); loss=loss_fn(logits,y)
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
                run_loss += loss.item()*x.size(0)
                correct += (logits.argmax(1)==y).sum().item()
                total += x.size(0)

            sched.step()
            va_res=evaluate_model(model,va)
            train_acc=correct/total
            row={"epoch":ep,"train_loss":run_loss/total,"train_acc":train_acc,
                 "val_loss":va_res["loss"],"val_acc":va_res["accuracy"],
                 "val_f1":va_res["macro_f1"],"minutes":(time.time()-t0)/60}
            history.append(row)
            print(f"{name} | ep {ep:02d} | train {train_acc:.3f} | val {va_res['accuracy']:.3f} | f1 {va_res['macro_f1']:.3f} | {row['minutes']:.1f}m")

            if va_res["accuracy"] > best_acc:
                best_acc=va_res["accuracy"]; best_state=deepcopy(model.state_dict()); bad=0
                torch.save(best_state,ckpt_path)
            else:
                bad+=1
                if bad>=patience:
                    print("Early stopping.")
                    break

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print("CUDA OOM:",name,"— skip this backbone.")
            return None

    if best_state is None: return None
    model.load_state_dict(best_state)
    test_res=evaluate_model(model,te)
    json.dump(history,open(ROOT/"reports"/f"{name}_history.json","w"),indent=2)

    report=classification_report(test_res["y_true"],test_res["y_pred"],target_names=class_names,output_dict=True)
    json.dump(report,open(ROOT/"reports"/f"{name}_classification_report.json","w"),indent=2)

    cm=confusion_matrix(test_res["y_true"],test_res["y_pred"])
    plt.figure(figsize=(max(12,NUM_CLASSES*.45),max(10,NUM_CLASSES*.40)))
    sns.heatmap(cm,xticklabels=class_names,yticklabels=class_names,cmap="Blues")
    plt.xticks(rotation=90); plt.yticks(rotation=0); plt.tight_layout()
    plt.savefig(ROOT/"reports"/f"{name}_confusion_matrix.png",dpi=180); plt.close()

    result={"name":name,"best_val_accuracy":best_acc,
            "test_accuracy":test_res["accuracy"],"test_macro_f1":test_res["macro_f1"],
            "checkpoint":str(ckpt_path),"input_size":size,"batch":batch}
    return result

candidate_models=["efficientnet_b0","efficientnet_b2"]
if VRAM_GB >= 12:
    candidate_models.append("efficientnet_v2_s")

results=[]
for model_name in candidate_models:
    r=train_one(model_name)
    if r is not None:
        results.append(r)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if not results:
    raise RuntimeError("Không có model nào train thành công.")

results_df=pd.DataFrame(results).sort_values(["best_val_accuracy","test_accuracy"],ascending=False)
results_df.to_csv(ROOT/"reports/model_comparison.csv",index=False)
print(results_df.to_string(index=False))


## 6. Chọn model và kiểm tra robustness

Model được chọn theo **validation accuracy**, sau đó mới đánh giá một lần trên test set độc lập trên các biến đổi giống camera. Đây không phải test set độc lập hoàn toàn; robustness test chỉ là phụ trợ.


In [ ]:
best_name = results_df.iloc[0]["name"]
best_size = int(results_df.iloc[0]["input_size"])
best_batch = int(results_df.iloc[0]["batch"])
best_ckpt = Path(results_df.iloc[0]["checkpoint"])

best_model = make_model(best_name, NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
best_model.eval()

print("BEST MODEL (selected by validation accuracy):", best_name)
print("Best validation:", results_df.iloc[0]["best_val_accuracy"])
print("Test accuracy:", results_df.iloc[0]["test_accuracy"])

# Confidence/coverage on test
_,_,best_test_loader=build_loaders(best_size,best_batch)
res=evaluate_model(best_model,best_test_loader)
conf=res["probs"].max(axis=1)

coverage_rows=[]
for th in [0.50,0.60,0.70,0.80,0.85,0.90,0.95]:
    keep=conf>=th
    if keep.sum():
        acc=accuracy_score(res["y_true"][keep],res["y_pred"][keep])
    else:
        acc=float("nan")
    coverage_rows.append({"threshold":th,"coverage":float(keep.mean()),"accepted_accuracy":float(acc)})
coverage=pd.DataFrame(coverage_rows)
coverage.to_csv(ROOT/"reports/confidence_coverage.csv",index=False)
print(coverage.to_string(index=False))


In [ ]:
# Camera-like corruption evaluation on a fixed test subset.
from PIL import ImageEnhance, ImageFilter

def corrupted(img, mode):
    if mode=="blur":
        return img.filter(ImageFilter.GaussianBlur(radius=1.2))
    if mode=="dark":
        return ImageEnhance.Brightness(img).enhance(0.55)
    if mode=="bright":
        return ImageEnhance.Brightness(img).enhance(1.35)
    if mode=="contrast":
        return ImageEnhance.Contrast(img).enhance(0.65)
    if mode=="sharp":
        return ImageEnhance.Sharpness(img).enhance(0.45)
    return img

# Rebuild a small deterministic dataset with corruption.
base_tf=transforms.Compose([
    transforms.Resize(int(best_size*256/224)),
    transforms.CenterCrop(best_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])

def eval_corruption(mode, limit_per_class=40):
    rows=[]
    for ci,sp in enumerate(class_names):
        files=list((ROOT/"data"/"test"/sp).glob("*.jpg"))[:limit_per_class]
        for f in files:
            try:
                im=Image.open(f).convert("RGB")
                im=corrupted(im,mode)
                x=base_tf(im).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    p=torch.softmax(best_model(x),1)[0].cpu().numpy()
                rows.append((ci,int(p.argmax()),float(p.max())))
            except Exception:
                pass
    if not rows: return {"accuracy":0,"count":0}
    yt=np.array([x[0] for x in rows]); yp=np.array([x[1] for x in rows])
    return {"accuracy":float((yt==yp).mean()),"count":len(rows)}

robust={}
for mode in ["blur","dark","bright","contrast","sharp"]:
    robust[mode]=eval_corruption(mode)
print(json.dumps(robust,indent=2))
json.dump(robust,open(ROOT/"reports/robustness.json","w"),indent=2)


## 7. Export ONNX

Export dynamic batch với input name `input`, output `output`. Metadata đi kèm để backend ASP.NET Core dùng đúng preprocessing.


In [ ]:
import onnx, onnxruntime as ort

export_path = ROOT/"exports"/f"{best_name}_v4.onnx"
dummy=torch.randn(1,3,best_size,best_size,device=DEVICE)

torch.onnx.export(
    best_model, dummy, export_path,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input":{0:"batch_size"},"output":{0:"batch_size"}},
    opset_version=17
)

m=onnx.load(str(export_path)); onnx.checker.check_model(m)
print("ONNX:", export_path, round(export_path.stat().st_size/1024/1024,2),"MB")

meta_out={
    "model_name":best_name,
    "input_name":"input",
    "output_name":"output",
    "input_size":best_size,
    "mean":IMAGENET_MEAN,
    "std":IMAGENET_STD,
    "class_to_idx":class_to_idx,
    "idx_to_class":{str(v):k for k,v in class_to_idx.items()},
    "best_val_accuracy":float(results_df.iloc[0]["best_val_accuracy"]),
    "test_accuracy":float(results_df.iloc[0]["test_accuracy"]),
    "selected_species":class_names,
    "dataset":"dbabnigg/botanical-vision-256",
    "seed":SEED
}
json.dump(meta_out,open(ROOT/"exports/model_metadata_v4.json","w"),indent=2)

# Verify ONNX prediction against PyTorch.
sess=ort.InferenceSession(str(export_path),providers=["CPUExecutionProvider"])
x=dummy[:1].detach().cpu().numpy()
with torch.no_grad():
    pt=best_model(dummy[:1]).detach().cpu().numpy()
ox=sess.run(["output"],{"input":x})[0]
print("max abs diff:",float(np.max(np.abs(pt-ox))))
print("same argmax:",int(pt.argmax())==int(ox.argmax()))


## 8. Video inference + temporal smoothing

Đây là phần dành cho app camera/video. Có thể upload `.mp4` vào Colab và notebook sẽ:
- lấy frame theo FPS;
- chạy classifier;
- dùng exponential moving average trên xác suất;
- chỉ công bố kết quả khi confidence đủ cao;
- trả `UNKNOWN` nếu không đủ tin cậy.

Điều này không biến accuracy ảnh thành 99%; nó chỉ làm kết quả video ổn định hơn.


In [ ]:
import cv2

video_model = best_model
video_model.eval()

video_tf = base_tf

def predict_frame(frame_bgr):
    rgb=cv2.cvtColor(frame_bgr,cv2.COLOR_BGR2RGB)
    im=Image.fromarray(rgb)
    x=video_tf(im).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p=torch.softmax(video_model(x),1)[0].cpu().numpy()
    idx=int(p.argmax())
    return class_names[idx], float(p[idx]), p

def run_video(video_path, sample_fps=3, alpha=0.35, threshold=0.75, max_frames=600):
    cap=cv2.VideoCapture(str(video_path))
    native_fps=cap.get(cv2.CAP_PROP_FPS) or 30
    step=max(1,int(round(native_fps/sample_fps)))
    ema=None; out=[]; frame_i=0

    while len(out)<max_frames:
        ok,frame=cap.read()
        if not ok: break
        if frame_i % step != 0:
            frame_i+=1; continue

        name,conf,p=predict_frame(frame)
        ema=p if ema is None else alpha*p+(1-alpha)*ema
        idx=int(ema.argmax())
        smooth_conf=float(ema[idx])
        shown=name if smooth_conf>=threshold else "UNKNOWN"

        out.append({
            "frame":frame_i,
            "raw_class":name,
            "raw_confidence":conf,
            "smooth_class":class_names[idx],
            "smooth_confidence":smooth_conf,
            "shown_result":shown
        })
        frame_i+=1

    cap.release()
    df=pd.DataFrame(out)
    return df

print("Function ready: run_video('/content/your_video.mp4')")


## 9. Gói artifact để tải về

Sau khi Run All hoàn tất, cell này tạo một ZIP chứa model ONNX, mapping, reports và notebook metadata.


In [ ]:
import zipfile, os

zip_path=Path("/content/plant_recognition_v4_artifacts.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in (ROOT/"exports").rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(ROOT))
    for p in (ROOT/"reports").rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(ROOT))
    for p in (ROOT/"metadata").rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(ROOT))

print("DONE:",zip_path)
print("Best model:",best_name)
print("Test accuracy:",float(results_df.iloc[0]["test_accuracy"]))
print("Classes:",NUM_CLASSES)
